**Imported Libs & Symbol Config**

In [ ]:
import requests
import time

import yfinance as yf
import pandas as pd



# Alpha Vantage

Chỉ được 25 call/ngày

## Fetch Daily Stock Data

---

Giá cổ phiếu lịch sử và Chỉ số kỹ thuật

In [ ]:
from google.colab import userdata
# for stalling
# Alphavantage doesnt let retrieving data too fast

watchlist_alpha = ['AAPL', 'MSFT', 'TSLA', 'BTC', 'ETH', 'SOL', 'EURUSD', 'USDJPY', 'WTI', 'GOLD']
Custom_symbol = ""
API_KEY = userdata.get('S_API')

if Custom_symbol == "":
  SYMBOL = watchlist_alpha[0]
else:
  if Custom_symbol not in watchlist_alpha:
    watchlist_alpha.append(Custom_symbol)
  SYMBOL = Custom_symbol

# This URL asks for "TIME_SERIES_DAILY" (Historical Daily Prices)
url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={SYMBOL}&apikey={API_KEY}'

# 2. Call the API (This is the "Integration" test)
print("Fetching data...")
response = requests.get(url)
data = response.json()

# 3. Print the results for the most recent day
# Alpha Vantage puts the data inside a dictionary called "Time Series (Daily)"
time_series = data.get("Time Series (Daily)", {})

# Check if time_series is empty, indicating an API key issue or invalid response
if not time_series:
    print("Error: Could not retrieve daily time series data. Please check your API key and ensure it's valid.")
    print(f"API Response: {data}")
else:
    # Get the first date available in the data
    latest_date = list(time_series.keys())[0]
    latest_data = time_series[latest_date]

    print(f"\n--- Data for {SYMBOL} on {latest_date} ---")
    print(f"Open: ${latest_data.get('1. open', 'N/A')}")
    print(f"High: ${latest_data.get('2. high', 'N/A')}")
    print(f"Low: ${latest_data.get('3. low', 'N/A')}")
    print(f"Close: ${latest_data.get('4. close', 'N/A')}")
    print(f"Volume: {latest_data.get('5. volume', 'N/A')} shares")


time.sleep(2)    #stall

# This asks Alpha Vantage to calculate the 20-day Average (SMA) for you
Timeperiod = 40
url = f'https://www.alphavantage.co/query?function=SMA&symbol=AAPL&interval=daily&time_period={Timeperiod}&series_type=close&apikey={API_KEY}'
r = requests.get(url)
data = r.json()

if "Technical Analysis: SMA" in data:
    latest_date = list(data["Technical Analysis: SMA"].keys())[0]
    print(f"20-Day Average (SMA) for {latest_date}:", data["Technical Analysis: SMA"][latest_date])
else:
    print("Error retrieving SMA data. API Response:", data)

print(f"-----------------------------------------\n")

Fetching data...
Error: Could not retrieve daily time series data. Please check your API key and ensure it's valid.
API Response: {'Information': 'We have detected your API key as NFLZA9SO0L7KUGSV and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.'}
Error retrieving SMA data. API Response: {'Information': 'We have detected your API key as NFLZA9SO0L7KUGSV and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.'}
-----------------------------------------



## Fetch News Sentiment

---
Tin tức tài chính
Đường link tới trang tin tức mới nhất


In [ ]:
# 1. Stall to respect the burst limit
time.sleep(2)

url = f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={SYMBOL}&apikey={API_KEY}'

print(f"Fetching news for {SYMBOL} from Alpha Vantage...")
response = requests.get(url)
data = response.json()

if "feed" in data:
    print(f"\n--- Latest News for {SYMBOL} ---")
    for article in data["feed"][:5]:
        title = article['title']
        source = article['source']
        label = article['overall_sentiment_label']
        score = article['overall_sentiment_score']
        url_link = article['url']

        # Outputting with the score
        print(f"- {title}")
        print(f"  Source: {source} | Label: {label} | Score: {score}")
        print(f"  URL: {url_link}\n")
else:
    print("Error retrieving SMA data. API Response:", data)

Fetching news for AAPL from Alpha Vantage...
Error retrieving SMA data. API Response: {'Information': 'We have detected your API key as NFLZA9SO0L7KUGSV and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.'}


### Fetch Daily Stock Data and SMA for all Watchlist Symbols (Alpha Vantage)

In [ ]:
# Helper function to fetch daily stock data from Alpha Vantage
def fetch_daily_alpha_vantage(symbol, api_key):
    url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&apikey={api_key}'
    response = requests.get(url)
    data = response.json()
    time_series = data.get("Time Series (Daily)", {})

    if not time_series:
        print(f"Error: Could not retrieve daily time series data for {symbol}. API Response: {data}")
        return None
    else:
        # Get the first date available in the data
        latest_date = list(time_series.keys())[0]
        latest_data = time_series[latest_date]
        return latest_date, latest_data

# Helper function to fetch SMA from Alpha Vantage
def fetch_sma_alpha_vantage(symbol, api_key, time_period=20):
    url = f'https://www.alphavantage.co/query?function=SMA&symbol={symbol}&interval=daily&time_period={time_period}&series_type=close&apikey={api_key}'
    response = requests.get(url)
    data = response.json()

    if "Technical Analysis: SMA" in data:
        latest_date = list(data["Technical Analysis: SMA"].keys())[0]
        return latest_date, data["Technical Analysis: SMA"][latest_date].get('SMA', 'N/A')
    else:
        print(f"Error retrieving SMA data for {symbol}. API Response: {data}")
        return None, None

# Iterate through the watchlist
for symbol_in_list in watchlist_alpha:
    print(f"\n--- Fetching data for {symbol_in_list} from Alpha Vantage ---")
    daily_result = fetch_daily_alpha_vantage(symbol_in_list, API_KEY)
    if daily_result:
        latest_date, latest_data = daily_result
        print(f"Daily Data for {symbol_in_list} on {latest_date}:")
        print(f"  Open: ${latest_data.get('1. open', 'N/A')}")
        print(f"  High: ${latest_data.get('2. high', 'N/A')}")
        print(f"  Low: ${latest_data.get('3. low', 'N/A')}")
        print(f"  Close: ${latest_data.get('4. close', 'N/A')}")
        print(f"  Volume: {latest_data.get('5. volume', 'N/A')} shares")

    # Small delay between daily and SMA call
    time.sleep(1)

    sma_date, sma_value = fetch_sma_alpha_vantage(symbol_in_list, API_KEY)
    if sma_value:
        print(f"  20-Day SMA for {symbol_in_list} on {sma_date}: ${float(sma_value):.2f}")

    print("-----------------------------------------")
    # Alpha Vantage has a rate limit of 5 calls per minute for the free tier.
    # We are making 2 calls per symbol (daily + SMA).


--- Fetching data for AAPL from Alpha Vantage ---
Error: Could not retrieve daily time series data for AAPL. API Response: {'Information': 'Thank you for using Alpha Vantage! Please consider spreading out your free API requests more sparingly (1 request per second). You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to lift the free key rate limit (25 requests per day), raise the per-second burst limit, and instantly unlock all premium endpoints'}
Error retrieving SMA data for AAPL. API Response: {'Information': 'We have detected your API key as NFLZA9SO0L7KUGSV and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.'}
-----------------------------------------

--- Fetching data for MSFT from Alpha Vantage ---
Error: Could not retrieve daily time series data for MSFT. API Response: {'Information': 'Thank you for using Alpha 

# Yahoo Finance

Không cần API key

In [ ]:
# Install the library (Run this cell once)
# !pip install yfinance --upgrade --quiet

## Giá cổ phiếu lịch sử và tính toán SMA

### Historical Stock Data for all Watchlist Symbols (Yahoo Finance)

In [ ]:
watchlist_yahoo = ["AAPL", "BTC-USD", "VNM.VN", "EURUSD=X"]
Custom_symbol = ""

if Custom_symbol != "" and (Custom_symbol not in watchlist_yahoo):
  watchlist_yahoo.append(Custom_symbol)

for symbol_in_list in watchlist_yahoo:
    print(f"\nFetching historical data for {symbol_in_list} from Yahoo Finance...")

    # Initialize the Ticker
    ticker = yf.Ticker(symbol_in_list)

    # Get the last 3 months of data (or adjust period as needed)
    hist_data = ticker.history(period="3mo")

    if not hist_data.empty:
        # Calculate SMA (example, assuming 20-day SMA is still desired)
        hist_data['SMA_20'] = hist_data['Close'].rolling(window=20).mean()

        # Extract the latest day's data
        latest_date = hist_data.index[-1].strftime('%Y-%m-%d')
        latest_data = hist_data.iloc[-1]

        print(f"--- Data for {symbol_in_list} on {latest_date} ---")
        print(f"Open: ${latest_data['Open']:.2f}")
        print(f"High: ${latest_data['High']:.2f}")
        print(f"Low: ${latest_data['Low']:.2f}")
        print(f"Close: ${latest_data['Close']:.2f}")
        print(f"Volume: {int(latest_data['Volume'])} shares")
        if 'SMA_20' in latest_data:
            print(f"Calculated 20-Day SMA: ${latest_data['SMA_20']:.2f}")
        print(f"-----------------------------------------")
    else:
        print(f"No historical data found for {symbol_in_list} in the last 3 months.")
    time.sleep(1)


Fetching historical data for AAPL from Yahoo Finance...
--- Data for AAPL on 2026-05-06 ---
Open: $281.92
High: $286.72
Low: $281.08
Close: $285.38
Volume: 19516787 shares
Calculated 20-Day SMA: $270.12
-----------------------------------------

Fetching historical data for BTC-USD from Yahoo Finance...
--- Data for BTC-USD on 2026-05-05 ---
Open: $79823.53
High: $81751.45
Low: $79787.58
Close: $80927.05
Volume: 39700107376 shares
Calculated 20-Day SMA: $77310.55
-----------------------------------------

Fetching historical data for VNM.VN from Yahoo Finance...
--- Data for VNM.VN on 2026-05-06 ---
Open: $61200.00
High: $61700.00
Low: $60700.00
Close: $61500.00
Volume: 2595810 shares
Calculated 20-Day SMA: $61575.00
-----------------------------------------

Fetching historical data for EURUSD=X from Yahoo Finance...
--- Data for EURUSD=X on 2026-05-06 ---
Open: $1.17
High: $1.18
Low: $1.17
Close: $1.18
Volume: 0 shares
Calculated 20-Day SMA: $1.17
-----------------------------------

## Tin tức tài chính

---
5 bài báo mới nhất (có thể thay đổi số lượng bài báo mới nhất)






In [ ]:
SYMBOL = watchlist_yahoo[0]
ticker = yf.Ticker(SYMBOL)
news = ticker.news

number_of_article = 10

for article in news[:number_of_article]:
    # The library only gave us 'id' and 'content'
    # So we look inside 'content'
    details = article.get('content', {})

    title = details.get('title', 'Still No Title')
    source = details.get('pubDate', 'No Date') # Sometimes they hide the source here
    link = details.get('canonicalUrl', {}).get('url', 'No Link')

    print(f"● {title}")
    print(f"  Link: {link}\n")

● Tech stocks today: AMD earnings buoy chip stocks, Musk-Altman court battle continues
  Link: https://finance.yahoo.com/sectors/technology/live/tech-stocks-today-semiconductor-earnings-ai-boom-musk-altman-fight-100000447.html

● Micron stock is surging, hitting a new intraday record high. Why?
  Link: https://finance.yahoo.com/video/micron-stock-is-surging-hitting-a-new-intraday-record-high-why-195211594.html

● Apple Explores New Chip Suppliers As Investors Weigh Supply Chain Risks
  Link: https://finance.yahoo.com/markets/stocks/articles/apple-explores-chip-suppliers-investors-151908786.html

● Apple to pay $250M to settle lawsuit over Siri’s delayed AI features
  Link: https://finance.yahoo.com/sectors/technology/articles/apple-pay-250m-settle-lawsuit-151249649.html

● Samsung Hits $1 Trillion As AI Chip Demand Fuels 14% Surge
  Link: https://finance.yahoo.com/markets/stocks/articles/samsung-hits-1-trillion-ai-145859204.html

● US Supreme Court declines to pause order holding Apple

# Reuter


In [ ]:
import requests
from google.colab import userdata

news_api = userdata.get('NEWS_API_KEY')
SYMBOL = 'AAPL'

# The previous query was too restrictive, only searching reuters.com.
# We will remove the 'domains' filter to search all available sources.
url = f'https://newsapi.org/v2/everything?q={SYMBOL}&apiKey={news_api}'

response = requests.get(url)
data = response.json()

if data.get('status') == 'ok':
    articles = data.get('articles', [])
    total = data.get('totalResults', 0)

    if total > 0:
        print(f"--- News Feed: {SYMBOL} ({total} results found) ---")
        for art in articles[:5]:
            print(f"● {art['title']}")
            print(f"  Source: {art['source']['name']} | Date: {art['publishedAt']}")
            print(f"  Link: {art['url']}\n")
    else:
        print(f"No articles found for '{SYMBOL}' in the last 30 days. Try a different symbol or check API key.")
elif data.get('status') == 'error':
    print(f"API ERROR: {data.get('message')}. Please check your NEWS_API_KEY in Colab secrets.")
else:
    print(f"Unexpected API response: {data}")

--- News Feed: AAPL (356 results found) ---
● What’s the consensus on $AAPL?
  Source: Asymco.com | Date: 2026-04-22T21:21:48Z
  Link: https://asymco.com/2026/04/22/whats-the-consensus-on-aapl/

● Apple Inc. (AAPL): Israel Englander Trims Stake
  Source: Yahoo Entertainment | Date: 2026-04-10T14:31:00Z
  Link: https://consent.yahoo.com/v2/collectConsent?sessionId=1_cc-session_d6ea46c4-17d7-47f3-a2f8-268b54d5d964

● Testing macOS on the Apple Network Server 2.0 ROMs
  Source: Blogspot.com | Date: 2026-05-03T15:49:33Z
  Link: http://oldvcr.blogspot.com/2026/05/testing-macos-on-apple-network-server.html

● Here’s Why Warren Buffett and Ken Griffin Love Apple (AAPL)
  Source: Yahoo Entertainment | Date: 2026-04-06T21:19:03Z
  Link: https://consent.yahoo.com/v2/collectConsent?sessionId=1_cc-session_a97a69e4-5bfb-4da6-b80f-e7dc35634fe5

● Apple Inc. (AAPL) Appointed John Ternus as CEO
  Source: Yahoo Entertainment | Date: 2026-04-25T05:57:23Z
  Link: https://consent.yahoo.com/v2/collectConse

### Fetch News for all Watchlist Symbols (NewsAPI)

In [ ]:
def fetch_news_for_symbol(symbol, api_key, num_articles=3):
    """
    Fetches news articles for a given symbol using the NewsAPI.
    """
    url = f'https://newsapi.org/v2/everything?q={symbol}&apiKey={api_key}'
    response = requests.get(url)
    data = response.json()

    if data.get('status') == 'ok':
        articles = data.get('articles', [])
        total = data.get('totalResults', 0)

        print(f"\n--- News Feed: {symbol} ({total} results found) ---")
        if total > 0:
            for i, art in enumerate(articles[:num_articles]):
                print(f"● {art['title']}")
                print(f"  Source: {art['source']['name']} | Date: {art['publishedAt']}")
                print(f"  Link: {art['url']}\n")
        else:
            print(f"No articles found for '{symbol}' in the last 30 days.")
    elif data.get('status') == 'error':
        print(f"API ERROR for {symbol}: {data.get('message')}. Please check your NEWS_API_KEY.")
    else:
        print(f"Unexpected API response for {symbol}: {data}")

# Assuming watchlist and news_api are already defined in the notebook
for symbol_in_list in watchlist:
    fetch_news_for_symbol(symbol_in_list, news_api)
    time.sleep(1) # Add a small delay to avoid hitting API rate limits too quickly


--- News Feed: AAPL (356 results found) ---
● What’s the consensus on $AAPL?
  Source: Asymco.com | Date: 2026-04-22T21:21:48Z
  Link: https://asymco.com/2026/04/22/whats-the-consensus-on-aapl/

● Apple Inc. (AAPL): Israel Englander Trims Stake
  Source: Yahoo Entertainment | Date: 2026-04-10T14:31:00Z
  Link: https://consent.yahoo.com/v2/collectConsent?sessionId=1_cc-session_d6ea46c4-17d7-47f3-a2f8-268b54d5d964

● Testing macOS on the Apple Network Server 2.0 ROMs
  Source: Blogspot.com | Date: 2026-05-03T15:49:33Z
  Link: http://oldvcr.blogspot.com/2026/05/testing-macos-on-apple-network-server.html


--- News Feed: BTC-USD (2655 results found) ---
● SatoshiGuesser – Roll for Bitcoin
  Source: Github.com | Date: 2026-04-30T16:29:58Z
  Link: https://github.com/Pathos0925/SatoshiGuesser

● Brazil Bans Stablecoins For Cross-Border Payments
  Source: Cryptoprowl.com | Date: 2026-05-03T18:16:00Z
  Link: https://www.cryptoprowl.com/releases/brazil-bans-stablecoins-for-cross-border-payments